# SAR-MIGRES — execution / control panel

State Aware Retrieval (SAR) replaces the document-retrieval decision inside MIGRES; MIGRES keeps its
filtering, evidence extraction and information-gap updates. **All logic lives in `src/sar_migres/`** — this
notebook only configures and runs it.

**Status: design + core implementation.** Nothing here evaluates SAR. Baselines, Bayesian optimisation,
ablations, RGB robustness and static-vs-dynamic weighting are *October* modules (section 16, disabled).

Quick start: run all cells as-is (`MODE = "local"`) for the offline smoke test — no API keys, no downloads.
For a real run set `MODE = "real"` after following the README.

## 0. Settings you are expected to change

In [ ]:
# ----------------------------------------------------------------------------------------------
# MODE "local": hashing embedder + fake LLM + offline stand-in loop  (validates plumbing only; NOT scientific)
# MODE "real" : configs/default.yaml -> Sentence Transformer + Pinecone + official MIGRES code + Gemini/OpenAI
# ----------------------------------------------------------------------------------------------
MODE            = "local"                  # "local" | "real"
EXTRA_CONFIGS   = []                       # extra YAML files merged on top, e.g. ["configs/my_experiment.yaml"]
OVERRIDES       = {}                       # e.g. {"sar": {"alpha": 0.5, "beta": 0.3, "gamma": 0.2, "k": 5},
                                           #       "llm": {"provider": "gemini", "model": "<your model id>"},
                                           #       "vector_store": {"type": "local"}}
QUESTION        = None                     # None -> first question of the configured dataset; or "your question?"
BATCH_LIMIT     = 3                        # number of dataset questions in the batch run (section 13)
RUN_BATCH       = True
SHOW_TEXT_CHARS = 200                      # snippet length in the trace tables
RUN_FUTURE_EXPERIMENTS = False             # section 16 - October stage, disabled by default

## 1. Environment / imports

In [ ]:
import json, os, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "sar_migres").exists():          # notebook started from notebooks/
    ROOT = ROOT.parent
assert (ROOT / "src" / "sar_migres").exists(), f"run from the project root or notebooks/ (looked in {ROOT})"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import sar_migres
from sar_migres.config import load_config, load_dotenv_file
from sar_migres.corpus import Corpus
from sar_migres.datasets import load_questions
from sar_migres.embeddings import build_embedder
from sar_migres.logging_utils import setup_logging
from sar_migres.pipeline import SARMigresPipeline
from sar_migres.vector_store import build_vector_store, sync_corpus_embeddings
try:
    import pandas as pd
except ImportError:
    pd = None
try:
    from IPython.display import display
except ImportError:
    display = print
print("sar_migres", sar_migres.__version__, "| python", sys.version.split()[0], "| project root:", ROOT)

## 2. Configuration

In [ ]:
cfg_files = ["configs/local.yaml"] if MODE == "local" else ["configs/default.yaml"]
cfg = load_config(*cfg_files, *EXTRA_CONFIGS, overrides=OVERRIDES)   # validates alpha+beta+gamma = 1, K >= 1, ...
setup_logging(cfg.run.log_level)
print("config files :", cfg_files + EXTRA_CONFIGS)
print("embedder     :", cfg.embedding.type, cfg.embedding.model_name if cfg.embedding.type == "sentence_transformer" else "(hashing stand-in)")
print("vector store :", cfg.vector_store.type)
print("SAR          : alpha=%.2f beta=%.2f gamma=%.2f K=%d  (configured values, not optimised)" % (cfg.sar.alpha, cfg.sar.beta, cfg.sar.gamma, cfg.sar.k))
print("MIGRES       :", cfg.migres.backend, "| max_iter =", cfg.migres.max_iter)
print("LLM          :", cfg.llm.provider)

## 3. API-key and dependency checks (names only — values are never printed)

In [ ]:
import importlib.util as iu
loaded = load_dotenv_file(ROOT / ".env")
print("variables loaded from .env:", loaded or "none (no .env file, or nothing new)")

need_env = []
if cfg.vector_store.type == "pinecone":            need_env.append(cfg.vector_store.pinecone_api_key_env)
if cfg.llm.provider == "gemini":                   need_env.append(cfg.llm.api_key_env or "GEMINI_API_KEY")
if cfg.llm.provider == "openai":                   need_env.append(cfg.llm.api_key_env or "OPENAI_API_KEY")
if cfg.llm.provider != "offline_deterministic" and not cfg.llm.model:
    need_env.append(cfg.llm.model_env)
if cfg.migres.backend == "upstream":               need_env.append("MIGRES_REPO_DIR") if not cfg.migres.repo_dir else None
need_pkgs = {"sentence_transformer": ["sentence_transformers"]}.get(cfg.embedding.type, [])
if cfg.vector_store.type == "pinecone": need_pkgs.append("pinecone")
if cfg.llm.provider == "gemini": need_pkgs.append("google.genai")
if cfg.llm.provider == "openai": need_pkgs.append("openai")
if cfg.migres.backend == "upstream" and cfg.migres.reranker == "upstream_cross_encoder": need_pkgs += ["torch", "transformers", "nltk"]

problems = []
for v in need_env:
    ok = bool(os.environ.get(v)); print(("OK      " if ok else "MISSING ") + "env  " + v); problems += [] if ok else [v]
for p in need_pkgs:
    try: ok = iu.find_spec(p) is not None
    except ModuleNotFoundError: ok = False
    print(("OK      " if ok else "MISSING ") + "pkg  " + p); problems += [] if ok else [p]
if cfg.migres.backend == "upstream":
    repo = cfg.resolve_migres_repo()
    ok = bool(repo) and (repo / "code" / "pipeline.py").exists(); print(("OK      " if ok else "MISSING ") + f"MIGRES clone: {repo}"); problems += [] if ok else ["MIGRES clone"]
assert not problems, f"Fix these first (see README 'Environment variables'): {problems}"
print("\nall prerequisites for MODE =", MODE, "are present")

## 4. Corpus configuration

In [ ]:
print("corpus file :", cfg.data.corpus_path, "| lazy loading:", cfg.data.corpus_lazy)
print("dataset     :", cfg.data.dataset, "->", cfg.data.dataset_path)

## 5. Load or prepare the corpus

In [ ]:
corpus = Corpus.from_file(cfg.data.corpus_path, embed_template=cfg.embedding.embed_template, lazy=cfg.data.corpus_lazy,
                          max_malformed_fraction=cfg.data.max_malformed_fraction)
print(f"{len(corpus)} documents | version {corpus.version[:16]}… | load report: {corpus.report.to_dict()}")

## 6. Build / load cached embeddings  ·  7. Initialise the vector store

In [ ]:
embedder = build_embedder(cfg.embedding)
print("embedder:", embedder.name, "| dim", embedder.dim, "| semantic model:", embedder.is_semantic)
store = build_vector_store(cfg.vector_store, embedder, cfg.run.cache_root)          # Pinecone (default) or local matrix
report = sync_corpus_embeddings(store, corpus, embedder, cfg.embedding.batch_size)   # new -> embed, changed -> re-embed, unchanged -> reuse
print("embedding sync:", report.to_dict())
print("re-running is a no-op:", sync_corpus_embeddings(store, corpus, embedder).n_embedded == 0)

## 8. Initialise the MIGRES adapter (LLM + backend + SAR retriever)

In [ ]:
pipe = SARMigresPipeline(cfg, corpus=corpus, embedder=embedder, store=store).prepare()
print("run id      :", pipe.run_id)
print("run dir     :", pipe.run_dir)
print("MIGRES      :", json.dumps(pipe.backend.describe(), indent=1, default=str)[:900])
print("LLM         :", pipe.llm.provider, "/", pipe.llm.model)
print()
for w in pipe.warnings: print("WARNING:", w)

## 9. Single-question smoke test

In [ ]:
item = ({"id": "adhoc", "question": QUESTION} if QUESTION
        else load_questions(cfg.data.dataset, cfg.data.dataset_path, 1)[0])
print("Q:", item["question"])
result = pipe.run_single(item)          # also writes config snapshot, JSONL/CSV traces and report.md to pipe.run_dir
print(f"\nstatus = {result.status} | final answer = {result.final_answer!r} | gold = {result.gold_answer!r} | {result.elapsed_seconds:.2f}s")
if result.error: print("error:", result.error)
if cfg.llm.provider == "offline_deterministic":
    print("(offline mode: the answer comes from a rule-based fake, it is not meaningful)")

## 10–12. Retrieval trace: state after every iteration, documents, R / C / N components and final scores

In [ ]:
from sar_migres.utils import snippet
for t in result.traces:
    after = t.migres_state_after
    print("=" * 110)
    print(f"ITERATION {t.iteration}   alpha={t.alpha:.2f} beta={t.beta:.2f} gamma={t.gamma:.2f}  K={t.k}   "
          f"{t.n_docs_scored} docs scored, {t.n_similarity_calcs} similarities, {t.total_seconds*1000:.1f} ms")
    print("  gaps     G_t :", t.gaps or "∅  (no gaps yet)")
    print("  evidence E_t :", t.evidence or "∅  (no evidence yet)")
    rows = []
    kept = set(after.get("filter_kept_doc_ids", []))
    for d in t.documents:
        rows.append({"rank": d["rank"], "doc_id": d["doc_id"], "title": d["title"], "R(d,q)": round(d["relevance"], 4),
                     "C(d,G)": round(d["gap_coverage"], 4), "N(d,E)": round(d["novelty"], 4), "S_t": round(d["score"], 4),
                     "passed MIGRES filter": (d["doc_id"] in kept) if "filter_kept_doc_ids" in after else "n/a",
                     "snippet": snippet(d["text"], SHOW_TEXT_CHARS)})
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for r in rows: print("  ", r)
    print("  -> MIGRES state after this retrieval: evidence added =", after.get("evidence_added", "n/a"),
          "| next gaps =", after.get("next_gaps", "n/a"))

## 13. Batch of dataset questions

In [ ]:
if RUN_BATCH:
    items = load_questions(cfg.data.dataset, cfg.data.dataset_path, BATCH_LIMIT)
    batch_pipe = SARMigresPipeline(cfg, corpus=corpus, embedder=embedder, store=store, llm=pipe.llm, backend=pipe.backend).prepare()
    results = batch_pipe.run_batch(items)
    summary = [{"id": r.question_id, "status": r.status, "iterations": len(r.traces), "answer": r.final_answer, "gold": r.gold_answer,
                "seconds": round(r.elapsed_seconds, 3)} for r in results]
    display(pd.DataFrame(summary)) if pd is not None else print("\n".join(map(str, summary)))
else:
    print("RUN_BATCH = False")

## 14. Machine-readable outputs (JSON / JSONL / CSV)  ·  15. Summary tables  ·  human-readable report

In [ ]:
if RUN_BATCH:
    out = batch_pipe.run_dir
    print("run directory:", out)
    for p in sorted(out.rglob("*")):
        if p.is_file() and "upstream_state" not in p.parts: print(f"  {p.relative_to(out)!s:<28} {p.stat().st_size:>9,d} bytes")
    print("\nsummary tables also copied to:", Path(cfg.run.output_dir) / "tables")
    print("\n--- traces.csv (head) ---")
    if pd is not None:
        display(pd.read_csv(out / "traces.csv").drop(columns=["run_id", "question", "evidence", "snippet"]).head(8))
    print("\n--- report.md (first 25 lines) ---")
    print("\n".join((out / "report.md").read_text().splitlines()[:25]))

## 16. FUTURE / OCTOBER experiments — disabled

These are **interfaces only**. Nothing below was run for this milestone, and no result from them is claimed.
Enabling the flag runs *nothing heavy*: it only demonstrates the plug-in points on the tiny synthetic corpus.

In [ ]:
if RUN_FUTURE_EXPERIMENTS:
    from sar_migres import baselines, evaluation
    from sar_migres.scoring import SARWeights
    print("Ablation weight settings (Sec. 5.6.3):", evaluation.AblationStudy.ablation_weights(SARWeights(cfg.sar.alpha, cfg.sar.beta, cfg.sar.gamma)))
    bm25 = baselines.BM25Baseline(corpus); print("BM25 baseline top-3 for the last question:", bm25.retrieve(item["question"], 3))
    print("Bayesian optimisation: sar_migres.optimization.optimize_weights(objective) — supply a validation-set objective (not provided).")
    print("Static vs dynamic weighting: dynamic rule not specified in methodology.tex -> NotImplemented by design.")
    print("RGB robustness: datasets.load_rgb() exists; evaluation.RGBRobustnessEvaluation is a stub.")
else:
    print("RUN_FUTURE_EXPERIMENTS = False  (October stage: evaluation, baselines, Bayesian optimisation, RGB, static-vs-dynamic)")